Legacy registry/parquet explorer. For exchange-run JSON artifacts use `notebooks/stage_B/stage_b_run_explorer.ipynb`. Retained for earlier pipeline runs.

# Run health QC

Read-only inspection of a pipeline run: unit status, failure classes, token spend,
rating distributions, and evidence spot checks. This notebook never calls the API.

In [ ]:
import json
import sqlite3
from pathlib import Path

import pandas as pd

REGISTRY = Path("registry.sqlite")
RUN_DIR = sorted(Path("runs").glob("stage_a_v1_*"))[-1]
print(RUN_DIR)

In [ ]:
with sqlite3.connect(REGISTRY) as conn:
    units = pd.read_sql_query("SELECT * FROM work_units", conn)
units.groupby(["pass_name", "status"]).size().unstack(fill_value=0)

## Failure classes — the raw material for the failure taxonomy

In [ ]:
failures = units[units.status == "failed"]
display(failures.groupby("error_class").size())
display(failures[["video_id", "k", "error_class", "error_detail"]].head(20))

## Token spend

In [ ]:
units[units.status == "done"][["input_tokens", "output_tokens"]].sum()

## Rating distributions and test–retest spread

In [ ]:
annotations = pd.read_parquet(RUN_DIR / "tables" / "annotations.parquet")
display(annotations.groupby("code_id")["value"].describe()[["count", "mean", "std", "min", "max"]])
display(
    annotations.groupby(["video_id", "code_id"])["value"].std()
    .groupby("code_id").mean().sort_values(ascending=False)
    .rename("mean within-video SD across replicates")
)

## Evidence spot check — are the model's justifications locatable?

In [ ]:
annotations.sample(min(15, len(annotations)), random_state=0)[
    ["video_id", "code_id", "value", "confidence", "evidence"]
]

## Manifest — the reproducibility record

In [ ]:
print(json.dumps(json.loads((RUN_DIR / "manifest.json").read_text()), indent=2))